# FinRisk — XGBoost Impact Model (v4 — Fixed)

## What was wrong in v3 and how this fixes it

**v3 problems:**
1. **Label noise** — daily returns used as AR labels because hourly data only covers 59 days. Daily returns are ~70% noise.
2. **False ticker matches** — `visa` matched travel articles, `apple` matched food articles → 14 tickers per article average.
3. **Too sparse** — 351 files covered the year too thinly.

**v4 fixes:**
1. **Earnings-anchored labels** — use yfinance `earnings_dates` as the primary label source. Earnings are precisely timestamped, ticker-specific, and always cause measurable price reactions. No alias noise.
2. **GDELT for non-earnings events** — sample more densely (1 of every 10 files), use strict ticker matching with exact company names + word boundaries.
3. **Intraday return window** — since earnings dates are available, we can compute 1-day forward returns from the announcement, avoiding the hourly data gap.
4. **Clean label construction** — separate the label distribution for earnings vs non-earnings events, then combine.

**Expected result:** Spearman ρ > 0.25, MAE < 1.5 (competitive with rule-based baseline)

**Runtime:** ~40 minutes on Colab free CPU

## Cell 1 — Install

In [1]:
!pip install -U \
    "xgboost>=2.1.0" \
    "yfinance>=0.2.50" \
    "scipy>=1.14.1" \
    "tqdm>=4.66.2" \
    "pyarrow>=17.0.0" \
    "pandas>=2.2.3" \
    "numpy>=2.1.0" \
    "pyyaml>=6.0.1" \
    "scikit-learn>=1.5.2"

print('Done.')

Done.


## Cell 2 — Configuration

In [2]:
from pathlib import Path
import json

DATA_DIR     = Path('/content/finrisk')
MODELS_DIR   = DATA_DIR / 'models'
GDELT_JSONL  = DATA_DIR / 'gdelt_rows.jsonl'
LABELED_OUT  = DATA_DIR / 'labeled_events.parquet'
PERCNT_OUT   = DATA_DIR / 'impact_percentiles.json'
MODEL_OUT    = MODELS_DIR / 'impact_xgb.pkl'
FEATURES_OUT = MODELS_DIR / 'impact_xgb_features.json'
for d in [DATA_DIR, MODELS_DIR]: d.mkdir(parents=True, exist_ok=True)

TICKERS          = ['AAPL','MSFT','NVDA','GOOGL','AMZN','META','TSLA',
                    'JPM','BAC','GS','V','JNJ','UNH','PFE',
                    'XOM','CVX','CAT','BA','WMT','HD','SPY']
UNIVERSE_TICKERS = [t for t in TICKERS if t != 'SPY']

EVENT_TYPES = ['Earnings','M&A','Regulatory','Macro','Geopolitical',
               'Credit','Commodity','Cyber','Leadership','Product']
SECTORS     = ['Technology','Communication Services','Consumer Discretionary',
               'Financials','Health Care','Energy','Industrials','Consumer Staples']
TICKER_SECTOR = {
    'AAPL':'Technology','MSFT':'Technology','NVDA':'Technology',
    'GOOGL':'Communication Services','META':'Communication Services',
    'AMZN':'Consumer Discretionary','TSLA':'Consumer Discretionary','HD':'Consumer Discretionary',
    'JPM':'Financials','BAC':'Financials','GS':'Financials','V':'Financials',
    'JNJ':'Health Care','UNH':'Health Care','PFE':'Health Care',
    'XOM':'Energy','CVX':'Energy',
    'CAT':'Industrials','BA':'Industrials',
    'WMT':'Consumer Staples',
}
BASE_IMPACT = {
    'Earnings':5.0,'M&A':5.5,'Regulatory':6.0,'Macro':6.0,
    'Geopolitical':7.0,'Credit':8.0,'Commodity':5.5,
    'Cyber':6.5,'Leadership':4.5,'Product':4.0,
}

# STRICT ticker aliases — only unambiguous company names
# Deliberately excludes 'apple' (fruit), 'visa' (travel), 'amazon' (rainforest)
# by requiring more specific multi-word names or capitalized forms
TICKER_ALIASES_STRICT = {
    'AAPL':  ['apple inc','apple computer','iphone maker','apple stock'],
    'MSFT':  ['microsoft','microsoft corp','microsoft corporation'],
    'NVDA':  ['nvidia','nvidia corp','nvidia corporation','nvda'],
    'GOOGL': ['alphabet inc','alphabet corp','google llc','googl','google search'],
    'AMZN':  ['amazon.com','amazon inc','amazon web services','aws amazon'],
    'META':  ['meta platforms','meta inc','facebook inc','instagram owner'],
    'TSLA':  ['tesla inc','tesla motors','tesla electric','elon musk tesla'],
    'JPM':   ['jpmorgan chase','jp morgan chase','jpmorgan','jpmorganchase'],
    'BAC':   ['bank of america','bankofamerica','bofa securities'],
    'GS':    ['goldman sachs','goldmansachs','goldman sachs group'],
    'V':     ['visa inc','visa corporation','visa payment','visainc'],
    'JNJ':   ['johnson & johnson','johnson and johnson','j&j consumer'],
    'UNH':   ['unitedhealth group','unitedhealth','united health group'],
    'PFE':   ['pfizer inc','pfizer corp','pfizer pharmaceutical'],
    'XOM':   ['exxonmobil','exxon mobil','exxon corporation'],
    'CVX':   ['chevron corporation','chevron corp','chevron oil'],
    'CAT':   ['caterpillar inc','caterpillar corp'],
    'BA':    ['boeing company','boeing corp','the boeing'],
    'WMT':   ['walmart inc','walmart stores','wal-mart'],
    'HD':    ['home depot','homedepot'],
}

# GDELT sampling: 1 of every 10 files = ~3,500 files, ~20 min
SAMPLE_EVERY_N = 10
LOOKBACK_DAYS  = 730  # 2 years for better coverage

print('Config ready.')
print(f'  Sampling 1/{SAMPLE_EVERY_N} files over {LOOKBACK_DAYS} days')
print(f'  Strict aliases: {sum(len(v) for v in TICKER_ALIASES_STRICT.values())} patterns for {len(TICKER_ALIASES_STRICT)} tickers')

Config ready.
  Sampling 1/10 files over 730 days
  Strict aliases: 67 patterns for 20 tickers


## Cell 3 — Download prices for ALL tickers (2 years daily)

**This cell runs FIRST** — we need prices before GDELT so we can validate
that AR labels are computable.

Downloads:
- 2 years of **daily** OHLCV for all 20 tickers + SPY (for abnormal returns)
- 59 days of **hourly** OHLCV for all tickers (short-window returns)
- `earnings_dates` for all tickers (primary label source)

**Expected: ~5 minutes**

In [3]:
import yfinance as yf
import pandas as pd
import time
from datetime import datetime, timedelta, timezone
from tqdm.notebook import tqdm

daily_start  = (datetime.now() - timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d')
hourly_start = (datetime.now() - timedelta(days=59)).strftime('%Y-%m-%d')

print(f'Downloading prices: daily from {daily_start}, hourly from {hourly_start}')

for ticker in tqdm(TICKERS, desc='Prices'):
    t = yf.Ticker(ticker)
    for interval, start, label in [('1d', daily_start,'daily'), ('1h', hourly_start,'1h')]:
        for attempt in range(3):
            try:
                df = t.history(start=start, interval=interval, auto_adjust=True)
                if not df.empty:
                    df.index = df.index.tz_convert('UTC') if df.index.tz else df.index.tz_localize('UTC')
                    df.index.name = 'datetime'
                    df.to_parquet(DATA_DIR / f'prices_{label}_{ticker}.parquet')
                break
            except Exception as e:
                if attempt < 2: time.sleep(3)
    time.sleep(0.4)

# Verify SPY loaded
spy_d = pd.read_parquet(DATA_DIR / 'prices_daily_SPY.parquet')
print(f'\n✓ SPY daily: {len(spy_d)} bars  {spy_d.index[0].date()} → {spy_d.index[-1].date()}')

# Download earnings dates for all universe tickers
print('\nDownloading earnings dates...')
earnings_data = {}
for ticker in tqdm(UNIVERSE_TICKERS, desc='Earnings'):
    try:
        t = yf.Ticker(ticker)
        ed = t.earnings_dates
        if ed is not None and not ed.empty:
            # earnings_dates index is timezone-aware
            ed.index = ed.index.tz_convert('UTC')
            earnings_data[ticker] = ed
    except Exception:
        pass
    time.sleep(0.3)

print(f'\n✓ Earnings data: {len(earnings_data)} tickers')
total_earnings = sum(len(v) for v in earnings_data.values())
print(f'  Total earnings events: {total_earnings:,}')
for ticker, df in list(earnings_data.items())[:3]:
    print(f'  {ticker}: {len(df)} quarters, latest {df.index[0].date()}')

Prices:   0%|          | 0/21 [00:00<?, ?it/s]


✓ SPY daily: 501 bars  2024-10-07 → 2026-10-06



Earnings:   0%|          | 0/20 [00:00<?, ?it/s]


✓ Earnings data: 20 tickers
  Total earnings events: 500
  AAPL: 25 quarters, latest 2026-10-29
  MSFT: 25 quarters, latest 2026-10-28
  NVDA: 25 quarters, latest 2026-11-17


## Cell 4 — Build earnings-anchored labels

**Why earnings first:**
- Precisely timestamped — no guessing when the event happened
- Single-ticker — no false matches
- Large price reactions — high signal-to-noise ratio
- ~80 events per ticker × 20 tickers = ~1,600 clean labels

Label = 1-day forward return (announcement day close → next day close) minus SPY return.
This uses **daily** prices which cover 2 years — no 59-day limit issue.

In [4]:
import pandas as pd
import numpy as np
from datetime import timezone


def _load_daily(ticker):
    p = DATA_DIR / f'prices_daily_{ticker}.parquet'
    if not p.exists(): return None
    df = pd.read_parquet(p)
    if df.index.tz is None: df.index = df.index.tz_localize('UTC')
    else: df.index = df.index.tz_convert('UTC')
    return df['Close'].sort_index()


def _forward_return(prices, event_ts, n_days=1):
    """
    Compute n-day forward return from first price AFTER event_ts.
    No look-ahead: entry_price is the first bar strictly after event_ts.
    """
    future = prices[prices.index > event_ts]
    if len(future) < n_days + 1: return None, None
    entry_ts    = future.index[0]
    entry_price = float(future.iloc[0])
    exit_price  = float(future.iloc[n_days])
    exit_ts     = future.index[n_days]
    assert entry_ts > event_ts  # no look-ahead
    ret = (exit_price - entry_price) / entry_price
    return ret, entry_ts


# Load prices
daily_px = {t: _load_daily(t) for t in UNIVERSE_TICKERS + ['SPY']}
spy_d    = daily_px['SPY']

# Build earnings records
earnings_records = []
for ticker, edf in earnings_data.items():
    stock_px = daily_px.get(ticker)
    if stock_px is None: continue

    for event_ts in edf.index:
        # 1-day forward return
        stock_ret, entry_ts = _forward_return(stock_px, event_ts, n_days=1)
        spy_ret,   _        = _forward_return(spy_d,    event_ts, n_days=1)
        if stock_ret is None or spy_ret is None: continue

        ar_1d = stock_ret - spy_ret

        # Also try 2-day return for robustness
        stock_ret2, _ = _forward_return(stock_px, event_ts, n_days=2)
        spy_ret2,   _ = _forward_return(spy_d,    event_ts, n_days=2)
        ar_2d = (stock_ret2 - spy_ret2) if (stock_ret2 is not None and spy_ret2 is not None) else ar_1d

        abs_ar = max(abs(ar_1d), abs(ar_2d))

        # Get EPS surprise if available (strong predictor)
        eps_surprise = None
        try:
            row = edf.loc[event_ts]
            reported = row.get('Reported EPS', row.get('EPS Actual', None))
            estimate = row.get('EPS Estimate', None)
            if reported is not None and estimate is not None and estimate != 0:
                eps_surprise = float((reported - estimate) / abs(estimate))
        except Exception:
            pass

        earnings_records.append({
            'article_ts':    event_ts,
            'ticker':        ticker,
            'event_source':  'earnings',
            'event_type_label': 'Earnings',
            'tone_overall':  0.0,   # filled later from EPS surprise
            'tone_positive': max(0, (eps_surprise or 0)) * 10,
            'tone_negative': max(0, -(eps_surprise or 0)) * 10,
            'ar_1d':         ar_1d,
            'abs_ar_max':    abs_ar,
            'eps_surprise':  eps_surprise,
            'is_after_hours':1.0,  # earnings are usually after market
            'source_count':  5.0,  # earnings always hit multiple sources
        })

earnings_df = pd.DataFrame(earnings_records)
print(f'✓ Earnings events: {len(earnings_df):,}')
print(f'  Tickers covered: {earnings_df["ticker"].nunique()}')
print(f'  Date range: {earnings_df["article_ts"].min().date()} → {earnings_df["article_ts"].max().date()}')
print(f'  Mean |AR|: {earnings_df["abs_ar_max"].mean():.4f} ({earnings_df["abs_ar_max"].mean()*100:.2f}%)')
print(f'  |AR| > 5%: {(earnings_df["abs_ar_max"] > 0.05).sum()} events')

✓ Earnings events: 480
  Tickers covered: 20
  Date range: 2020-10-13 → 2026-08-26
  Mean |AR|: 0.0198 (1.98%)
  |AR| > 5%: 30 events


## Cell 5 — Fetch GDELT for non-earnings events

Strict matching: only match when a full unambiguous company name appears  
in Organizations column. Reject single-word aliases that cause false matches.

**Memory-safe:** streams directly to disk.

**Expected: ~20 minutes** (3,500 files at 1/10 sampling)

In [5]:
import io
import json
import time
import gc
import zipfile

from pathlib import Path
from datetime import datetime, timedelta, timezone

import requests
from tqdm.notebook import tqdm


# ── Configuration ────────────────────────────────────────────────────────────
MAX_RETRIES = 2
GDELT_BASE = 'http://data.gdeltproject.org/gdeltv2'


# Make sure required variables exist
if 'LOOKBACK_DAYS' not in globals():
    LOOKBACK_DAYS = 30

if 'SAMPLE_EVERY_N' not in globals():
    SAMPLE_EVERY_N = 24

if 'GDELT_JSONL' not in globals():
    GDELT_JSONL = Path('/content/finrisk/gdelt_rows.jsonl')

GDELT_JSONL = Path(GDELT_JSONL)

GDELT_JSONL.parent.mkdir(
    parents=True,
    exist_ok=True
)


print(f'LOOKBACK_DAYS = {LOOKBACK_DAYS}')
print(f'SAMPLE_EVERY_N = {SAMPLE_EVERY_N}')
print(f'GDELT_JSONL = {GDELT_JSONL}')


# ── HTTP downloader ──────────────────────────────────────────────────────────
def _get(url, timeout=35):

    for i in range(MAX_RETRIES):

        try:

            r = requests.get(
                url,
                timeout=timeout
            )

            if r.status_code == 200:
                return r

            if r.status_code == 404:
                return None

        except requests.RequestException:

            pass

        if i < MAX_RETRIES - 1:
            time.sleep(2)

    return None


# ── Build flat list of (alias, ticker) pairs ─────────────────────────────────
# Longer aliases first → more specific matches before shorter ones.
_ALIAS_PAIRS = sorted(
    [
        (alias.lower(), ticker)
        for ticker, aliases in TICKER_ALIASES_STRICT.items()
        for alias in aliases
        if alias and ticker
    ],
    key=lambda x: -len(x[0])
)


print(
    f'Loaded {len(_ALIAS_PAIRS):,} company aliases.'
)


# ── Strict ticker matching ────────────────────────────────────────────────────
def _match_tickers_strict(
    orgs,
    v2orgs,
    url
):
    """
    Strict matching.

    Primary:
      Organizations  (col 13)
      V2Organizations (col 14)

    Secondary:
      Article URL (col 4), but ONLY for multi-word aliases.

    Returns:
      Deduplicated list of ticker symbols.
    """

    # Convert:
    # Apple Inc,2;Microsoft Corporation,5
    #
    # into:
    # apple inc microsoft corporation

    v2n = ' '.join(
        p.split(',')[0]
        for p in v2orgs.split(';')
        if p.strip()
    )

    org_text = (
        orgs + ' ' +
        v2n
    ).lower()

    url_text = url.lower()

    found = set()

    for alias, ticker in _ALIAS_PAIRS:

        if ticker in found:
            continue

        # Primary match: Organizations / V2Organizations
        if alias in org_text:

            found.add(ticker)

        # Secondary URL match:
        # only allow aliases containing multiple words
        elif (
            len(alias.split()) >= 2
            and alias in url_text
        ):

            found.add(ticker)

    return list(found)


# ── Parse GKG and write matching rows ────────────────────────────────────────
def _parse_and_write(
    raw_bytes,
    fh_out
):

    written = 0

    try:

        text = raw_bytes.decode(
            'utf-8',
            errors='replace'
        )

        for line in text.splitlines():

            if not line.strip():
                continue

            f = line.split('\t')

            # Need at least 17 columns
            if len(f) < 17:
                continue

            orgs = f[13]
            v2orgs = f[14]
            url = f[4]

            tickers = _match_tickers_strict(
                orgs,
                v2orgs,
                url
            )

            if not tickers:
                continue

            # ── V2Tone, column 16 ────────────────────────────────────────────
            tp = f[16].split(',')

            try:

                t_ov = float(tp[0])

                t_po = (
                    float(tp[1])
                    if len(tp) > 1
                    else 0.0
                )

                t_ne = (
                    float(tp[2])
                    if len(tp) > 2
                    else 0.0
                )

            except (ValueError, IndexError):

                t_ov = 0.0
                t_po = 0.0
                t_ne = 0.0

            # ── Output record ───────────────────────────────────────────────
            row = {
                'DATE': f[1],

                'SourceCommonName': f[3],

                'DocumentIdentifier': url,

                'V2Themes': (
                    f[8]
                    if len(f) > 8
                    else ''
                ),

                'Organizations': orgs,

                'V2Organizations': v2orgs,

                'tone_overall': t_ov,

                'tone_positive': t_po,

                'tone_negative': t_ne,

                'matched_tickers': tickers,
            }

            fh_out.write(
                json.dumps(
                    row,
                    ensure_ascii=False
                ) + '\n'
            )

            written += 1

    except Exception as e:

        # Don't kill the entire run because of one malformed row/file.
        return written

    return written


# ── Build GDELT URL list ──────────────────────────────────────────────────────
end_dt = datetime.now(
    tz=timezone.utc
)

start_dt = (
    end_dt -
    timedelta(days=LOOKBACK_DAYS)
)


print(
    '\nFetching master file list...'
)


resp = _get(
    'http://data.gdeltproject.org/gdeltv2/masterfilelist.txt',
    timeout=90
)


urls = []


if resp:

    for line in resp.text.splitlines():

        parts = line.split()

        if (
            len(parts) >= 3
            and parts[2].endswith('.gkg.csv.zip')
        ):

            u = parts[2].strip()

            fname = (
                u
                .split('/')[-1]
                .replace(
                    '.gkg.csv.zip',
                    ''
                )
            )

            try:

                fdt = datetime.strptime(
                    fname,
                    '%Y%m%d%H%M%S'
                ).replace(
                    tzinfo=timezone.utc
                )

                if start_dt <= fdt <= end_dt:
                    urls.append(u)

            except ValueError:

                pass


    urls = sorted(urls)


else:

    print(
        'WARNING: Could not fetch GDELT master file list.'
    )


print(
    f'Files in range: {len(urls):,}'
)


# ── Validate sample interval ──────────────────────────────────────────────────
if SAMPLE_EVERY_N < 1:

    raise ValueError(
        'SAMPLE_EVERY_N must be >= 1'
    )


sampled = urls[::SAMPLE_EVERY_N]


print(
    f'Sampling 1/{SAMPLE_EVERY_N} '
    f'→ {len(sampled):,} files to process'
)


if not sampled:

    raise RuntimeError(
        'No GDELT files selected. '
        'Check LOOKBACK_DAYS and the master file list.'
    )


# ── Process GDELT files ──────────────────────────────────────────────────────
total_rows = 0
skipped = 0
bad_zip = 0


with open(
    GDELT_JSONL,
    'w',
    encoding='utf-8'
) as fh_out:

    for i, url in enumerate(
        tqdm(
            sampled,
            desc='GDELT',
            unit='file'
        )
    ):

        r = _get(
            url,
            timeout=40
        )

        if r is None:

            skipped += 1
            continue

        csv_bytes = None

        try:

            # IMPORTANT:
            # .gkg.csv.zip = ZIP archive, NOT GZIP.
            with zipfile.ZipFile(
                io.BytesIO(r.content)
            ) as z:

                names = z.namelist()

                if not names:

                    skipped += 1
                    continue

                # Locate the GKG CSV inside the ZIP.
                csv_name = next(
                    (
                        name
                        for name in names
                        if name.endswith('.gkg.csv')
                    ),
                    names[0]
                )

                with z.open(csv_name) as f:

                    csv_bytes = f.read()


            # Parse and write matching rows
            n = _parse_and_write(
                csv_bytes,
                fh_out
            )

            total_rows += n

        except zipfile.BadZipFile:

            bad_zip += 1

        except Exception:

            skipped += 1

        finally:

            del r

            if csv_bytes is not None:
                del csv_bytes


        # Flush periodically
        if i % 200 == 0:

            fh_out.flush()
            gc.collect()


# ── Final statistics ──────────────────────────────────────────────────────────
try:

    file_mb = (
        GDELT_JSONL.stat().st_size /
        1e6
    )

except Exception:

    file_mb = 0.0


print(
    f'\n✓ GDELT done: '
    f'{total_rows:,} rows, '
    f'{file_mb:.1f} MB, '
    f'{skipped:,} skipped, '
    f'{bad_zip:,} bad ZIPs'
)


# ── Sanity check ──────────────────────────────────────────────────────────────
if total_rows == 0:

    raise RuntimeError(
        '0 matching rows. '
        'Run the GDELT diagnostic cell to verify '
        'TICKER_ALIASES_STRICT and the GDELT columns.'
    )


# ── Sample ticker distribution ───────────────────────────────────────────────
sample_matches = {}
match_count = 0


with open(
    GDELT_JSONL,
    encoding='utf-8'
) as fh:

    for line in fh:

        try:

            row = json.loads(line)

        except json.JSONDecodeError:

            continue

        for ticker in row.get(
            'matched_tickers',
            []
        ):

            sample_matches[ticker] = (
                sample_matches.get(ticker, 0) + 1
            )

            match_count += 1

        if match_count > 500:
            break


print(
    '\nSample ticker distribution '
    '(first 500 matches):'
)


for ticker, count in sorted(
    sample_matches.items(),
    key=lambda x: -x[1]
):

    print(
        f'  {ticker:6s} {count}'
    )


LOOKBACK_DAYS = 730
SAMPLE_EVERY_N = 10
GDELT_JSONL = /content/finrisk/gdelt_rows.jsonl
Loaded 67 company aliases.

Fetching master file list...
Files in range: 68,398
Sampling 1/10 → 6,840 files to process


GDELT:   0%|          | 0/6840 [00:00<?, ?file/s]


✓ GDELT done: 234,844 rows, 482.5 MB, 0 skipped, 0 bad ZIPs

Sample ticker distribution (first 500 matches):
  GOOGL  205
  MSFT   80
  NVDA   69
  GS     54
  JPM    38
  XOM    11
  AMZN   8
  AAPL   8
  META   6
  TSLA   5
  PFE    5
  CVX    4
  UNH    2
  HD     2
  WMT    2
  BAC    1
  CAT    1


## Cell 6 — Build GDELT abnormal return labels + combine with earnings

For GDELT events: use 1-day daily forward return (covers 2 years, no 59-day limit).  
Deduplicate: if multiple GDELT articles about the same ticker on the same day,
keep only the one with the largest |tone_overall|.  
Then combine with earnings labels and compute unified percentile bins.

In [6]:
import pandas as pd
import numpy as np
import json, gc
from datetime import datetime, timezone
from tqdm.notebook import tqdm

# Count GDELT rows
n_gdelt = sum(1 for _ in open(GDELT_JSONL))
print(f'Processing {n_gdelt:,} GDELT rows...')


def _forward_return_daily(prices, event_ts, n_days=1):
    """Daily forward return, strictly after event_ts."""
    future = prices[prices.index > event_ts]
    if len(future) < n_days + 1: return None
    entry_p = float(future.iloc[0])
    exit_p  = float(future.iloc[n_days])
    assert future.index[0] > event_ts
    return (exit_p - entry_p) / entry_p if entry_p else None


# Build interim records: (date, ticker) -> best row by |tone|
# This deduplication prevents the same story appearing 50x
dedup: dict = {}   # key: (date_str, ticker) -> row dict

with open(GDELT_JSONL) as fh:
    for line in tqdm(fh, total=n_gdelt, desc='Dedup GDELT'):
        if not line.strip(): continue
        try: row = json.loads(line)
        except: continue
        try:
            ts = pd.Timestamp(
                datetime.strptime(str(row['DATE']),'%Y%m%d%H%M%S').replace(tzinfo=timezone.utc)
            )
        except: continue
        date_str = ts.strftime('%Y-%m-%d')
        for ticker in row.get('matched_tickers',[]):
            key = (date_str, ticker)
            tone_abs = abs(row.get('tone_overall',0))
            if key not in dedup or tone_abs > abs(dedup[key]['tone_overall']):
                dedup[key] = {
                    'article_ts': ts, 'ticker': ticker,
                    'tone_overall': row.get('tone_overall',0.),
                    'tone_positive': row.get('tone_positive',0.),
                    'tone_negative': row.get('tone_negative',0.),
                    'V2Themes': row.get('V2Themes',''),
                    'SourceCommonName': row.get('SourceCommonName',''),
                    'DocumentIdentifier': row.get('DocumentIdentifier',''),
                }

print(f'Unique (date, ticker) pairs after dedup: {len(dedup):,}')

# Compute abnormal returns for deduplicated events
gdelt_records = []
la_errors = 0
for (date_str, ticker), row in tqdm(dedup.items(), desc='AR computation'):
    stock_px = daily_px.get(ticker)
    if stock_px is None: continue
    ts = row['article_ts']
    try:
        sr = _forward_return_daily(stock_px, ts, n_days=1)
        mr = _forward_return_daily(spy_d,    ts, n_days=1)
        sr2= _forward_return_daily(stock_px, ts, n_days=2)
        mr2= _forward_return_daily(spy_d,    ts, n_days=2)
    except AssertionError:
        la_errors += 1; continue
    if sr is None or mr is None: continue
    ar_1d = sr - mr
    ar_2d = (sr2 - mr2) if (sr2 is not None and mr2 is not None) else ar_1d
    abs_ar = max(abs(ar_1d), abs(ar_2d))

    gdelt_records.append({
        'article_ts':    ts,
        'ticker':        ticker,
        'event_source':  'gdelt',
        'event_type_label': None,  # inferred from V2Themes in Cell 7
        'tone_overall':  row['tone_overall'],
        'tone_positive': row['tone_positive'],
        'tone_negative': row['tone_negative'],
        'ar_1d':         ar_1d,
        'abs_ar_max':    abs_ar,
        'eps_surprise':  None,
        'is_after_hours':0.,
        'source_count':  1.,
        'V2Themes':      row['V2Themes'],
        'SourceCommonName': row['SourceCommonName'],
        'DocumentIdentifier': row['DocumentIdentifier'],
    })

if la_errors: print(f'  Look-ahead violations skipped: {la_errors}')
del dedup; gc.collect()

# Add V2Themes to earnings records too
for r in earnings_records:
    if 'V2Themes' not in r: r['V2Themes'] = 'earn;revenue;quarterly'
    if 'SourceCommonName' not in r: r['SourceCommonName'] = 'earnings_report'
    if 'DocumentIdentifier' not in r: r['DocumentIdentifier'] = ''

# Combine
all_records = gdelt_records + earnings_records
df = pd.DataFrame(all_records).sort_values('article_ts').reset_index(drop=True)

print(f'\nCombined dataset:')
print(f'  GDELT events:    {len(gdelt_records):,}')
print(f'  Earnings events: {len(earnings_records):,}')
print(f'  Total:           {len(df):,}')
print(f'  Mean |AR|:       {df["abs_ar_max"].mean():.4f} ({df["abs_ar_max"].mean()*100:.2f}%)')
print(f'  |AR| > 3%:       {(df["abs_ar_max"]>0.03).sum():,} events ({(df["abs_ar_max"]>0.03).mean()*100:.1f}%)')
print(f'  |AR| > 5%:       {(df["abs_ar_max"]>0.05).sum():,} events ({(df["abs_ar_max"]>0.05).mean()*100:.1f}%)')
print(f'\nPer-ticker counts:')
print(df['ticker'].value_counts().to_string())

Processing 234,844 GDELT rows...


Dedup GDELT:   0%|          | 0/234844 [00:00<?, ?it/s]

Unique (date, ticker) pairs after dedup: 9,848


AR computation:   0%|          | 0/9848 [00:00<?, ?it/s]


Combined dataset:
  GDELT events:    9,834
  Earnings events: 480
  Total:           10,314
  Mean |AR|:       0.0206 (2.06%)
  |AR| > 3%:       2,040 events (19.8%)
  |AR| > 5%:       608 events (5.9%)

Per-ticker counts:
ticker
GS       737
MSFT     737
GOOGL    737
NVDA     737
JPM      728
XOM      719
AMZN     707
AAPL     645
TSLA     637
META     624
UNH      604
WMT      510
CVX      456
PFE      398
HD       363
CAT      347
V        340
BAC      200
BA        62
JNJ       26


## Cell 7 — Build impact labels + feature engineering + XGBoost

Key improvements:
- **EPS surprise** as a direct feature for earnings events
- **Separate percentile bins** for earnings vs GDELT (earnings have higher |AR| on average)
- **source_count** and **is_after_hours** as meaningful features
- Same time-split: 70/15/15

In [7]:
import pickle
import pandas as pd, numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor


# ── Impact labels ─────────────────────────────────────────────────────────────
# Use COMBINED percentile distribution (not separate)
# This way earnings and GDELT events are on the same scale
PCTILE_BINS = [(0.,10.,(1,2)),(10.,30.,(3,4)),(30.,60.,(4,6)),(60.,80.,(6,8)),(80.,100.,(8,10))]

abs_ar = df['abs_ar_max'].values.astype(float)
p995   = float(np.percentile(abs_ar, 99.5))
clipped = np.clip(abs_ar, 0, p995)
thresholds = [float(np.percentile(clipped, p)) for p,_,_ in PCTILE_BINS[1:]]

print('Impact label percentile thresholds:')
for (p,_,_),t in zip(PCTILE_BINS[1:],thresholds):
    print(f'  {p:.0f}th pctile → |AR| > {t:.4f} ({t*100:.2f}%)')

def _map_impact(v):
    for i,(_,_,(lo,hi)) in enumerate(PCTILE_BINS):
        th = thresholds[i] if i < len(thresholds) else float('inf')
        if v < th:
            st = thresholds[i-1] if i > 0 else 0.
            bw = th - st
            fr = (v-st)/bw if bw > 0 else 0.5
            return float(lo) + fr*(hi-lo)
    lo,hi = PCTILE_BINS[-1][2]
    return float(lo) + min((v-thresholds[-1])/(thresholds[-1]+1e-9),1.)*(hi-lo)

df['impact_label'] = df['abs_ar_max'].apply(_map_impact).clip(1.,10.).round(2)

print(f'\nImpact label distribution:')
bins = [1,2,3,4,5,6,7,8,9,10.1]
lbls = ['1-2','2-3','3-4','4-5','5-6','6-7','7-8','8-9','9-10']
counts,_ = np.histogram(df['impact_label'].values, bins=bins)
for lbl,cnt in zip(lbls,counts):
    print(f'  {lbl}  {cnt:5d}  {"█"*min(cnt//(max(1,len(df)//40)),40)}')

PERCNT_OUT.write_text(json.dumps({'thresholds':thresholds,'p99_5':p995},indent=2))
df.to_parquet(LABELED_OUT, index=False)
print(f'\n✓ Saved {len(df):,} labeled events')


# ── Event type inference ──────────────────────────────────────────────────────
def _event_type(row):
    if row.get('event_type_label') == 'Earnings': return 'Earnings'
    t = str(row.get('V2Themes','')).lower()
    if any(k in t for k in ['merger','acquisition','takeover','buyout']): return 'M&A'
    if any(k in t for k in ['regulation','antitrust','penalty','fine','lawsuit']): return 'Regulatory'
    if any(k in t for k in ['federal_reserve','interest_rate','inflation','gdp','unemployment']): return 'Macro'
    if any(k in t for k in ['sanction','war','tariff','military','geopolit']): return 'Geopolitical'
    if any(k in t for k in ['credit','debt','downgrade','bankrupt','default']): return 'Credit'
    if any(k in t for k in ['oil','gas','commodity','opec','crude']): return 'Commodity'
    if any(k in t for k in ['cyber','hack','breach','ransomware']): return 'Cyber'
    if any(k in t for k in ['ceo','cfo','resign','appoint','executive']): return 'Leadership'
    if any(k in t for k in ['product','launch','recall','patent','drug']): return 'Product'
    # For GDELT rows without strong theme signal, use tone direction
    if abs(row.get('tone_overall',0)) > 3: return 'Earnings'  # treat as general news
    return 'Macro'

df['event_type_inferred'] = df.apply(_event_type, axis=1)


# ── Feature engineering ───────────────────────────────────────────────────────
def _is_after_hours(ts):
    try:
        t = pd.Timestamp(ts)
        t = t.tz_localize('UTC') if t.tz is None else t.tz_convert('UTC')
        return t.weekday() >= 5 or not (
            t.replace(hour=13,minute=30,second=0) <= t < t.replace(hour=20,minute=0,second=0))
    except: return False

out = pd.DataFrame(index=df.index)
out['sentiment']      = (df['tone_overall']/10.).clip(-1.,1.)
out['abs_sentiment']  = out['sentiment'].abs()
out['tone_positive']  = df['tone_positive'].fillna(0.)
out['tone_negative']  = df['tone_negative'].fillna(0.).abs()
out['source_count']   = df['source_count'].fillna(1.)
out['is_after_hours'] = df['is_after_hours'].fillna(0.)
# EPS surprise — very strong predictor for earnings events
out['eps_surprise']   = df['eps_surprise'].fillna(0.).clip(-2.,2.)
out['has_eps']        = df['eps_surprise'].notna().astype(float)
out['is_earnings']    = (df['event_type_label'] == 'Earnings').astype(float)

# Days since last event per ticker
ds = df.sort_values(['ticker','article_ts']).copy()
ds['prev_ts'] = ds.groupby('ticker')['article_ts'].shift(1)
ds['dsl'] = ((ds['article_ts']-ds['prev_ts']).dt.total_seconds()/86400).fillna(30.).clip(0.,90.)
out['days_since_last'] = ds['dsl'].reindex(df.index).fillna(30.)

# Event type one-hot
et_col = df['event_type_inferred']
for et in EVENT_TYPES:
    out[f"et_{et.lower().replace('&','and').replace(' ','_')}"] = (et_col==et).astype(float)

# Sector one-hot
sec_col = df['ticker'].map(TICKER_SECTOR).fillna('Technology')
for sec in SECTORS:
    out[f"sec_{sec.lower().replace(' ','_')}"] = (sec_col==sec).astype(float)

feat_cols = list(out.columns)
print(f'\nFeatures ({len(feat_cols)}): {feat_cols}')

X = out.values.astype(float)
y = df['impact_label'].values.astype(float)


# ── Time-based split ──────────────────────────────────────────────────────────
n = len(df)
t0,t1 = int(n*.70), int(n*.85)
X_tr,y_tr = X[:t0],y[:t0]
X_v, y_v  = X[t0:t1],y[t0:t1]
X_te,y_te = X[t1:],y[t1:]
print(f'\nSplit: train={len(X_tr):,}  val={len(X_v):,}  test={len(X_te):,}')
print(f'  Train ends: {df.iloc[t0]["article_ts"].date()}')
print(f'  Test  from: {df.iloc[t1]["article_ts"].date()}')


# ── XGBoost ───────────────────────────────────────────────────────────────────
model = XGBRegressor(
    objective='reg:squarederror', n_estimators=300, max_depth=5,
    learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
    min_child_weight=10,  # prevent overfitting on small groups
    random_state=42, n_jobs=-1, eval_metric='rmse',
    early_stopping_rounds=25, verbosity=0,
)
model.fit(X_tr,y_tr,eval_set=[(X_v,y_v)],verbose=False)
print(f'Best iteration: {model.best_iteration}')

y_pred = np.clip(model.predict(X_te),1.,10.)

# Rule-based baseline
et_te  = df['event_type_inferred'].iloc[t1:].values
s_te   = out['sentiment'].iloc[t1:].values
base   = np.clip([BASE_IMPACT.get(e,5.)*(0.5+0.5*abs(s))*0.7*0.9 for e,s in zip(et_te,s_te)],1.,10.)

xgb_mae  = mean_absolute_error(y_te,y_pred)
base_mae = mean_absolute_error(y_te,base)
xgb_rho, _  = spearmanr(y_te,y_pred)
base_rho,_  = spearmanr(y_te,base)

print(f'\n{"="*56}')
print(f'  {"Model":<28} {"MAE":>6} {"RMSE":>6} {"Spearman ρ":>11}')
print(f'  {"-"*54}')
rmse_base = float(np.sqrt(mean_squared_error(y_te,base)))
rmse_xgb  = float(np.sqrt(mean_squared_error(y_te,y_pred)))
print(f'  {"Rule-based baseline":<28} {base_mae:6.3f} {rmse_base:6.3f} {base_rho:+11.3f}')
print(f'  {"XGBoost v4":<28} {xgb_mae:6.3f} {rmse_xgb:6.3f} {xgb_rho:+11.3f}')
print(f'{"="*56}')
improvement = (base_mae-xgb_mae)/base_mae*100 if base_mae > 0 else 0
print(f'  MAE improvement: {improvement:+.1f}%  |  Spearman improvement: {xgb_rho-base_rho:+.3f}')

print('\nTop 10 feature importances:')
for feat,imp in sorted(zip(feat_cols,model.feature_importances_),key=lambda x:-x[1])[:10]:
    print(f'  {feat:<40} {imp:.4f}  {"█"*int(imp*60)}')


# ── Save ──────────────────────────────────────────────────────────────────────
MODELS_DIR.mkdir(parents=True, exist_ok=True)
with open(MODEL_OUT,'wb') as f: pickle.dump(model,f,protocol=pickle.HIGHEST_PROTOCOL)
meta = {
    'feature_columns':feat_cols, 'event_types':EVENT_TYPES, 'sectors':SECTORS,
    'n_train':int(len(X_tr)), 'n_val':int(len(X_v)), 'n_test':int(len(X_te)),
    'best_iteration':int(model.best_iteration),
    'test_mae':float(xgb_mae), 'test_rmse':float(rmse_xgb), 'test_spearman_rho':float(xgb_rho),
    'baseline_mae':float(base_mae), 'baseline_rmse':float(rmse_base), 'baseline_spearman_rho':float(base_rho),
    'version': 'v4',
}
FEATURES_OUT.write_text(json.dumps(meta,indent=2))
print(f'\n✓ Model saved: {MODEL_OUT}')

Impact label percentile thresholds:
  10th pctile → |AR| > 0.0050 (0.50%)
  30th pctile → |AR| > 0.0101 (1.01%)
  60th pctile → |AR| > 0.0193 (1.93%)
  80th pctile → |AR| > 0.0297 (2.97%)

Impact label distribution:
  1-2   1020  ███
  2-3     12  
  3-4   2053  ███████
  4-5   1718  ██████
  5-6   1380  █████
  6-7   1229  ████
  7-8    834  ███
  8-9   2068  ████████
  9-10      0  

✓ Saved 10,314 labeled events

Features (28): ['sentiment', 'abs_sentiment', 'tone_positive', 'tone_negative', 'source_count', 'is_after_hours', 'eps_surprise', 'has_eps', 'is_earnings', 'days_since_last', 'et_earnings', 'et_manda', 'et_regulatory', 'et_macro', 'et_geopolitical', 'et_credit', 'et_commodity', 'et_cyber', 'et_leadership', 'et_product', 'sec_technology', 'sec_communication_services', 'sec_consumer_discretionary', 'sec_financials', 'sec_health_care', 'sec_energy', 'sec_industrials', 'sec_consumer_staples']

Split: train=7,219  val=1,547  test=1,548
  Train ends: 2026-02-08
  Test  from: 2026

## Cell 8 — Download model files

In [8]:
from google.colab import files
import json

files.download(str(MODEL_OUT))
files.download(str(FEATURES_OUT))
files.download(str(PERCNT_OUT))

print('\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print('  EVALUATION RESULTS')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
m = json.loads(FEATURES_OUT.read_text())
print(f'  Training samples:    {m["n_train"]:,}')
print(f'  Test samples:        {m["n_test"]:,}')
print(f'  Baseline MAE:        {m["baseline_mae"]:.3f}')
print(f'  XGBoost MAE:         {m["test_mae"]:.3f}')
print(f'  MAE improvement:     {(m["baseline_mae"]-m["test_mae"])/m["baseline_mae"]*100:+.1f}%')
print(f'  Baseline Spearman ρ: {m["baseline_spearman_rho"]:+.3f}')
print(f'  XGBoost Spearman ρ:  {m["test_spearman_rho"]:+.3f}')
print(f'  Best iteration:      {m["best_iteration"]}')
print('━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print()
print('Place the 3 files in finrisk/data/models/')
print('Then add MODEL_IMPACT=xgboost to finrisk/.env')
print()
print('Verify locally:')
print('  cd finrisk')
print('  .venv/bin/python -c "')
print('  from app.engine.xgboost_model import XGBoostImpactModel')
print('  m = XGBoostImpactModel()')
print('  ok = m.load()')
print('  print(ok, m.predict(m.get_feature_vector({}, -0.7, \\"Geopolitical\\")))')
print('  "')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  EVALUATION RESULTS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Training samples:    7,219
  Test samples:        1,548
  Baseline MAE:        3.831
  XGBoost MAE:         1.754
  MAE improvement:     +54.2%
  Baseline Spearman ρ: -0.067
  XGBoost Spearman ρ:  +0.128
  Best iteration:      23
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Place the 3 files in finrisk/data/models/
Then add MODEL_IMPACT=xgboost to finrisk/.env

Verify locally:
  cd finrisk
  .venv/bin/python -c "
  from app.engine.xgboost_model import XGBoostImpactModel
  m = XGBoostImpactModel()
  ok = m.load()
  print(ok, m.predict(m.get_feature_vector({}, -0.7, \"Geopolitical\")))
  "
